In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [26]:
# Load essays.csv
essays = pd.read_csv('essays.csv', usecols=['projectid', 'short_description', 'essay'])

# Simple text features: length of short description and full essay
essays['short_desc_len'] = essays['short_description'].fillna('').apply(len)
essays['essay_len'] = essays['essay'].fillna('').apply(len)

# Keep just the features
essay_features = essays[['projectid', 'short_desc_len', 'essay_len']]

# ---- Feature Matrix from outcomes.csv ---- #

# Load outcomes.csv
outcomes = pd.read_csv('outcomes.csv')

In [27]:
binary_cols = [
    'at_least_1_teacher_referred_donor',
    'great_chat',
    'three_or_more_non_teacher_referred_donors',
    'one_non_teacher_referred_donor_giving_100_plus',
    'donation_from_thoughtful_donor',
    'is_exciting'
]
binary_cols

['at_least_1_teacher_referred_donor',
 'great_chat',
 'three_or_more_non_teacher_referred_donors',
 'one_non_teacher_referred_donor_giving_100_plus',
 'donation_from_thoughtful_donor',
 'is_exciting']

In [28]:
for col in binary_cols:
    outcomes[col] = outcomes[col].map({'t': 1, 'f': 0})
outcome_features = outcomes[['projectid',
                             'at_least_1_teacher_referred_donor',
                             'teacher_referred_count',
                             'non_teacher_referred_count',
                             'great_chat',
                             'three_or_more_non_teacher_referred_donors',
                             'one_non_teacher_referred_donor_giving_100_plus',
                             'donation_from_thoughtful_donor',
                             'is_exciting']]


In [31]:
df = essay_features.merge(outcome_features, on='projectid', how='inner').dropna()



In [32]:
df

,projectid,short_desc_len,essay_len,at_least_1_teacher_referred_donor,teacher_referred_count,non_teacher_referred_count,great_chat,three_or_more_non_teacher_referred_donors,one_non_teacher_referred_donor_giving_100_plus,donation_from_thoughtful_donor,is_exciting
1,ffffac55ee02a49d1abc87ba6fc61135,195,1155,0.0,0.0,7.0,0,1.0,0.0,0.0,0
2,ffff97ed93720407d70a2787475932b0,270,1327,0.0,0.0,3.0,1,1.0,1.0,0.0,0
3,ffff418bb42fad24347527ad96100f81,270,1081,0.0,0.0,1.0,1,0.0,0.0,0.0,0
4,ffff2d9c769c8fb5335e949c615425eb,199,2799,1.0,6.0,2.0,1,0.0,1.0,0.0,1
5,fffeebf4827d745aa36b17c2d38d1966,190,1212,0.0,0.0,1.0,0,0.0,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...
619321,0000ee613c92ddc5298bf63142996a5c,199,2375,1.0,2.0,4.0,0,1.0,1.0,0.0,0
619322,0000b38bbc7252972f7984848cf58098,190,2036,0.0,0.0,2.0,0,0.0,1.0,0.0,0
619323,00002d691c05c51a5fdfbb2baef0ba25,275,1273,0.0,0.0,5.0,1,1.0,0.0,0.0,0
619324,00002bff514104264a6b798356fdd893,273,1698,0.0,0.0,2.0,0,0.0,1.0,0.0,0


In [33]:
print(df.isnull().mean())

projectid                                         0.0
short_desc_len                                    0.0
essay_len                                         0.0
at_least_1_teacher_referred_donor                 0.0
teacher_referred_count                            0.0
non_teacher_referred_count                        0.0
great_chat                                        0.0
three_or_more_non_teacher_referred_donors         0.0
one_non_teacher_referred_donor_giving_100_plus    0.0
donation_from_thoughtful_donor                    0.0
is_exciting                                       0.0
dtype: float64


In [35]:
y = df['is_exciting']

# Features from essays
X_essay = df[['short_desc_len', 'essay_len']]

# Features from outcomes (carefully selected to avoid leakage)
X_outcome = df[[
    'at_least_1_teacher_referred_donor',
    'teacher_referred_count',
    'non_teacher_referred_count',
    'great_chat',
    'three_or_more_non_teacher_referred_donors',
    'one_non_teacher_referred_donor_giving_100_plus',
    'donation_from_thoughtful_donor'
]]


In [36]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_essay, y, test_size=0.3, random_state=42)
X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(X_outcome, y, test_size=0.3, random_state=42)


In [37]:
def evaluate_decision_tree(X_train, X_test, y_train, y_test, feature_set_name):
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_pred)
    print(f"--- Decision Tree evaluation ({feature_set_name}) ---")
    print(report)
    print(f"ROC-AUC: {auc_score}\n")

In [38]:
evaluate_decision_tree(X_train_e, X_test_e, y_train_e, y_test_e, 'Essay-based Features')

# Evaluate outcome-based features
evaluate_decision_tree(X_train_o, X_test_o, y_train_o, y_test_o, 'Outcome-based Features')

--- Decision Tree evaluation (Essay-based Features) ---
              precision    recall  f1-score   support

           0       0.93      0.99      0.96    146598
           1       0.06      0.01      0.02     10881

    accuracy                           0.92    157479
   macro avg       0.49      0.50      0.49    157479
weighted avg       0.87      0.92      0.89    157479

ROC-AUC: 0.49904126950209526

--- Decision Tree evaluation (Outcome-based Features) ---
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    146598
           1       0.90      0.98      0.94     10881

    accuracy                           0.99    157479
   macro avg       0.95      0.99      0.97    157479
weighted avg       0.99      0.99      0.99    157479

ROC-AUC: 0.9855617858579876

